In [30]:
import pandas as pd

In [31]:
hist_data = pd.read_csv("/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/src/outgoing/forecast/forecast_dataset.csv")
forecast_data = pd.read_csv("/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/src/outgoing/forecast/sku_forecast_poisson.csv")


In [32]:
hist_data

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00,11-20,3,11,False
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00,11-21,4,11,False
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00,11-22,5,11,False
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00,11-23,6,11,False


In [33]:
forecast_data.rename({"qty_pred": "qty"}, axis=1, inplace=True)

In [8]:
hist_data[["date", "qty", "tavg", "prcp", "tsun", "sale_percent"]]

,date,qty,tavg,prcp,tsun,sale_percent
0,2024-01-26,2.0,9.0,1.2,0.0,0.24
1,2024-01-27,0.0,3.0,0.0,510.0,0.24
2,2024-01-28,0.0,3.7,0.0,516.0,0.24
3,2024-01-29,2.0,5.5,0.0,492.0,0.24
4,2024-01-30,1.0,6.1,0.0,192.0,0.24
...,...,...,...,...,...,...
149851,2025-11-20,0.0,2.3,0.0,59.0,0.00
149852,2025-11-21,0.0,-1.0,0.0,136.0,0.00
149853,2025-11-22,0.0,-4.2,0.0,243.0,0.00
149854,2025-11-23,0.0,-2.6,2.5,155.0,0.00


,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00,11-20,3,11,False
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00,11-21,4,11,False
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00,11-22,5,11,False
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00,11-23,6,11,False


In [36]:
full_df = pd.concat([forecast_data[["date", "SKU", "qty", "tavg", "prcp", "tsun", "sale_percent"]], hist_data[["date", "SKU", "qty", "tavg", "prcp", "tsun", "sale_percent"]]]).sort_values("date").reset_index(drop=True)

In [37]:
import pandas as pd

def add_weather_label(
    df: pd.DataFrame,
    rain_prcp_mm: float = 1.0,    # >= 1 mm = rainy day
    sun_max_prcp_mm: float = 0.2, # almost no rain
    sun_tsun_min: int = 420       # >= 7 hours of sun (7 * 60)
) -> pd.DataFrame:
    """
    Expects columns: ['tavg', 'prcp', 'tsun'] (Meteostat daily data).
    Adds a 'weather_label' column with values in {'rain', 'normal', 'sun'}.
    """
    df = df.copy()
    
    # start with "normal"
    labels = pd.Series("normal", index=df.index)

    # 1) rainy days dominate: if enough precipitation, mark as rain
    labels[df["prcp"] >= rain_prcp_mm] = "rain"

    # 2) sunny days: very little rain + a lot of sunshine
    labels[(df["prcp"] <= sun_max_prcp_mm) & (df["tsun"] >= sun_tsun_min)] = "sun"

    df["weather_label"] = labels
    return df

# example usage:
# weather_df = add_weather_label(weather_df)


In [38]:
full_df = add_weather_label(full_df)

In [39]:
full_df["sale_active"] = (full_df["sale_percent"] > 0).astype(bool)

In [40]:
full_df

,date,SKU,qty,tavg,prcp,tsun,sale_percent,weather_label,sale_active
0,2024-01-26,92076,0.000000,9.00,1.20,0.0,0.00,rain,False
1,2024-01-26,ga11001,0.000000,9.00,1.20,0.0,0.00,rain,False
2,2024-01-26,ga10171,0.000000,9.00,1.20,0.0,0.24,rain,True
3,2024-01-26,9107ub,0.000000,9.00,1.20,0.0,0.24,rain,True
4,2024-01-26,9102ub,2.000000,9.00,1.20,0.0,0.24,rain,True
...,...,...,...,...,...,...,...,...,...
172475,2026-03-04,9112fz,0.003080,6.25,0.05,234.0,0.00,normal,False
172476,2026-03-04,9112ub,0.011824,6.25,0.05,234.0,0.00,normal,False
172477,2026-03-04,9114dx,0.001478,6.25,0.05,234.0,0.00,normal,False
172478,2026-03-04,92056,0.009522,6.25,0.05,234.0,0.00,normal,False


In [41]:
full_df

,date,SKU,qty,tavg,prcp,tsun,sale_percent,weather_label,sale_active
0,2024-01-26,92076,0.000000,9.00,1.20,0.0,0.00,rain,False
1,2024-01-26,ga11001,0.000000,9.00,1.20,0.0,0.00,rain,False
2,2024-01-26,ga10171,0.000000,9.00,1.20,0.0,0.24,rain,True
3,2024-01-26,9107ub,0.000000,9.00,1.20,0.0,0.24,rain,True
4,2024-01-26,9102ub,2.000000,9.00,1.20,0.0,0.24,rain,True
...,...,...,...,...,...,...,...,...,...
172475,2026-03-04,9112fz,0.003080,6.25,0.05,234.0,0.00,normal,False
172476,2026-03-04,9112ub,0.011824,6.25,0.05,234.0,0.00,normal,False
172477,2026-03-04,9114dx,0.001478,6.25,0.05,234.0,0.00,normal,False
172478,2026-03-04,92056,0.009522,6.25,0.05,234.0,0.00,normal,False


In [42]:
daily = (
    full_df.groupby("date", as_index=False)
      .agg({
          "qty": "sum",          # total units sold per day
          "tavg": "first",       # same weather per date → take first (or "mean")
          "prcp": "first",
          "tsun": "first",
          "sale_percent": "max", # highest discount that day
          "weather_label": "first",  # same label per date → take first
          "sale_active": "any",  # True if any row that day had a sale
      })
)

In [43]:
daily

,date,qty,tavg,prcp,tsun,sale_percent,weather_label,sale_active
0,2024-01-26,80.000000,9.00,1.20,0.0,0.24,rain,True
1,2024-01-27,5.000000,3.00,0.00,510.0,0.24,sun,True
2,2024-01-28,0.000000,3.70,0.00,516.0,0.24,sun,True
3,2024-01-29,56.000000,5.50,0.00,492.0,0.24,sun,True
4,2024-01-30,30.000000,6.10,0.00,192.0,0.24,normal,True
...,...,...,...,...,...,...,...,...
764,2026-02-28,11.816001,3.85,0.00,333.0,0.00,normal,False
765,2026-03-01,9.938566,4.20,0.05,174.0,0.00,normal,False
766,2026-03-02,23.392341,5.30,0.00,252.0,0.00,normal,False
767,2026-03-03,19.983880,4.55,0.00,282.0,0.00,normal,False


In [45]:
full_df

,date,SKU,qty,tavg,prcp,tsun,sale_percent,weather_label,sale_active
0,2024-01-26,92076,0.000000,9.00,1.20,0.0,0.00,rain,False
1,2024-01-26,ga11001,0.000000,9.00,1.20,0.0,0.00,rain,False
2,2024-01-26,ga10171,0.000000,9.00,1.20,0.0,0.24,rain,True
3,2024-01-26,9107ub,0.000000,9.00,1.20,0.0,0.24,rain,True
4,2024-01-26,9102ub,2.000000,9.00,1.20,0.0,0.24,rain,True
...,...,...,...,...,...,...,...,...,...
172475,2026-03-04,9112fz,0.003080,6.25,0.05,234.0,0.00,normal,False
172476,2026-03-04,9112ub,0.011824,6.25,0.05,234.0,0.00,normal,False
172477,2026-03-04,9114dx,0.001478,6.25,0.05,234.0,0.00,normal,False
172478,2026-03-04,92056,0.009522,6.25,0.05,234.0,0.00,normal,False


In [46]:
full_df.to_csv("/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/src/outgoing/forecast/sku_forecast_poisson_full.csv", index=False)

In [49]:
df = full_df

In [57]:

df = pd.read_csv("/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/src/outgoing/forecast/sku_forecast_poisson_full.csv")
df["date"] = pd.to_datetime(df["date"])

In [73]:
def get_sku_forecast(sku, start_date, end_date):
    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    mask = (df["date"] >= start_date) & (df["date"] <= end_date) & (df["SKU"] == sku)
    return df.loc[mask, ["date", "qty", "tavg", "sale_percent", "sale_active", "weather_label"]]

In [ ]:
get_sku_forecast()

In [77]:
start = datetime(year=2025, month=11, day=29)
end = datetime(year=2026, month=2, day=27)
sku = "9101DX"

In [79]:
get_sku_forecast(sku.lower(), start, end)

,date,qty,tavg,sale_percent,sale_active,weather_label
151118,2025-11-29,0.091177,6.00,0.2,True,rain
151369,2025-11-30,0.079382,5.20,0.2,True,rain
151566,2025-12-01,0.144547,3.80,0.0,False,normal
151791,2025-12-02,0.127058,5.10,0.0,False,normal
152015,2025-12-03,0.110924,5.80,0.0,False,normal
...,...,...,...,...,...,...
170382,2026-02-23,0.347302,8.40,0.0,False,normal
170606,2026-02-24,0.304191,6.65,0.0,False,normal
170831,2026-02-25,0.269758,5.15,0.0,False,normal
171055,2026-02-26,0.230282,4.30,0.0,False,normal


In [74]:
df

,date,SKU,qty,tavg,prcp,tsun,sale_percent,weather_label,sale_active
0,2024-01-26,92076,0.000000,9.00,1.20,0.0,0.00,rain,False
1,2024-01-26,ga11001,0.000000,9.00,1.20,0.0,0.00,rain,False
2,2024-01-26,ga10171,0.000000,9.00,1.20,0.0,0.24,rain,True
3,2024-01-26,9107ub,0.000000,9.00,1.20,0.0,0.24,rain,True
4,2024-01-26,9102ub,2.000000,9.00,1.20,0.0,0.24,rain,True
...,...,...,...,...,...,...,...,...,...
172475,2026-03-04,9112fz,0.003080,6.25,0.05,234.0,0.00,normal,False
172476,2026-03-04,9112ub,0.011824,6.25,0.05,234.0,0.00,normal,False
172477,2026-03-04,9114dx,0.001478,6.25,0.05,234.0,0.00,normal,False
172478,2026-03-04,92056,0.009522,6.25,0.05,234.0,0.00,normal,False


In [67]:
from datetime import datetime

In [72]:
pd.Timestamp(datetime.now().date())

Timestamp('2025-11-29 00:00:00')